## Imports

In [2]:
# TODO: Replace with your own username / path. HACK: This is a hack to make sure the local repo is recognized when using the UV kernel
import sys
USERNAME = None
PROJECT_ROOT = f'/src/omnimouse' # f'/home/{USERNAME}/workspace/omnimouse'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import os
import rootutils
from datetime import datetime
from copy import deepcopy
from pprint import pprint
from IPython.display import HTML, display
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import hydra
import torch
import lightning as L
from lightning.pytorch.loggers import Logger

from omnimouse.utils import (
    SessionMap,
    SessionMetadata,
    print_model_summary,
    ModelCheckpoint,
    omegaconf_to_dict,
    compose_omnimouse_config,
    get_rng_state,
)
from omnimouse.modeling import ModelArgs, Model, build_model
from omnimouse.tasks.task_utils import (
    setup_cuda,
    log_to_loggers,
    run_evaluation,
    add_overall_metrics,
    compile_model,
    parse_validation_dataset_cfgs,
    setup_dataloaders_single_rank,
)
from omnimouse.masking import MaskingStrategy
from omnimouse.masking.visualization import visualize_masking_strategies


# Setup root directory
root = rootutils.setup_root('..', indicator=".project-root", pythonpath=True)

In [4]:
import logging
from omnimouse.utils import RankedLogger
log = RankedLogger(__name__, rank_zero_only=True)
stdout_handler = logging.StreamHandler(sys.stdout)
stdout_handler.setLevel(logging.DEBUG)
    
log.logger.addHandler(stdout_handler)
log.info(f'Logger initialized!')

In [ ]:
log.hasHandlers()

### Config

In [ ]:
# Pick a pretrained OmniMouse checkpoint from the 🤗 Hub and its matching experiment config.
# Available scales: "1M" | "5M" | "20M" | "80M" | "300M".
from huggingface_hub import snapshot_download

SCALE = "80M"
SERVER_NAME = None # use default with `None`
EXPERIMENT_OVERRIDE = f"omnimouse_{SCALE}" # use default with `None`
EVAL_EXPERIMENT_OVERRIDE = None # "iclr"

# Downloads the rank_*.ckpt shards from the corresponding `the-enigma-project/omnimouse-<SCALE>`
# model repo into the local HF cache and returns the resolved directory path. Re-running after
# the first download is ~instant.
CHECKPOINT_PATH = snapshot_download(
    repo_id=f"the-enigma-project/omnimouse-{SCALE}",
    allow_patterns=["rank_*.ckpt"],
)
RUN_NAME = f"inference-{datetime.now().strftime('%m-%d-%Y-%H_%M')}"

output_dir, cfg = compose_omnimouse_config(
    project_root=root,
    config_name="eval",
    run_name=RUN_NAME,
    local_override=SERVER_NAME,
    experiment_override=EXPERIMENT_OVERRIDE,
    eval_experiment_override=EVAL_EXPERIMENT_OVERRIDE,
    ckpt_path=CHECKPOINT_PATH,
    print_config=False, # Set to `False` to avoid printing the config to the notebook (WARNING: It can be pretty long :)
    # TODO: You can add additional CLI overrides here (see commented examples below)
    cli_overrides=[
        # "trainer.limit_train_batches=100",
        # "model.num_self_attends_per_block=12",
    ]
)

### MLFlow Logging

We use MLFlow to keep track of run stats and compare metrics. You can use this in the notebook too!

To configure MLFlow, make sure you have the following keys in your `.env` file:

```
MLFLOW_TRACKING_URI=https://mlflow.enigmatic.stanford.edu/
MLFLOW_TRACKING_USERNAME=mlflow-runner 
MLFLOW_TRACKING_PASSWORD=""  # Ask someone from the team for the enigma MLFlow password
```

In [ ]:
# TODO: Enable/disable logging
ENABLE_LOGGING = True

logger = None
if ENABLE_LOGGING:
    logger: Logger = hydra.utils.instantiate(cfg.trainer.logger)
    print(f"Logging to MLFlow ({os.path.join(
        logger._tracking_uri,'#', 'experiments', logger.experiment_id, 'runs', logger.run_id
    )}")

### Seed

In [ ]:
# Set seed for random number generators in pytorch, numpy and python.random
seed = cfg.get("seed", None)
if seed is not None:
    L.seed_everything(seed, workers=True)
# Get initial random state to use for evaluation consistency
fix_eval_rng_state = cfg.get("fix_eval_rng_state", False)
initial_rng_state = get_rng_state() if fix_eval_rng_state else None

### Device

In [ ]:
# Configure CUDA backend settings for precision, sdp backend, and determinism
cuda_is_available = setup_cuda(
    enable_cudnn_sdp=True, deterministic=None, benchmark=None,
)

# TODO: Set this to select which GPU to use
CUDA_VISIBLE_DEVICES = 0

device = (
    torch.device(f"cuda:{CUDA_VISIBLE_DEVICES}") if cuda_is_available else 
    torch.device("cpu")
)

### Dataloaders

#### Choose dataset(s) and Build Dataloaders

##### `dataset_collection`
Specifies which datasets to use for training/evaluation. You have two options:

**Option 1: Predefined Collection Names (String)**
Pass a **string name** that references a predefined collection from `sandbox.py` or `scaling_laws.py`:

- **Small-scale**: `"single_demo"`, `"quick_2"`, `"public_5"`, `"initial_32"`
- **Large-scale**: `"pretraining_316"`, `"colossus"`, `"foundation_32"`

```python
dataset_collection = "public_5"  # Uses predefined 5-dataset collection
```

**Option 2: Custom List**
Pass your own list of dataset identifiers:

```python
dataset_collection = [
    'dynamic29234-6-9-Video-021a75e56847d574b9acbcc06c675055_30hz',
    'dynamic29514-2-9-Video-021a75e56847d574b9acbcc06c675055_30hz',
    'dynamic29228-2-10-Video-021a75e56847d574b9acbcc06c675055_30hz'
]
```

##### `validation_datasets`
Selects a subset of datasets from `dataset_collection` from which to pull validation splits. Configured the same way as `dataset_collection`, and will default to the entire `dataset_collection`:

```python
validation_datasets = "validation_5"  # Predefined validation collection
# OR
validation_datasets = ['dynamic29234-6-9-Video-021a75e56847d574b9acbcc06c675055_30hz']  # Custom list
```

**Note:** Some predefined collections are dictionaries (designed for multinode training) that are automatically flattened into a single list for notebook usage.

**Note:** In the container, available datasets on your device should be found in the mounted directory `/data/mouse`.

In [ ]:
# Define dataset collection to use for training
dataset_collection = [
    'dynamic29156-11-10-Video-021a75e56847d574b9acbcc06c675055_30hz',
    'dynamic29513-3-5-Video-021a75e56847d574b9acbcc06c675055_30hz',
]

data_cfg = OmegaConf.create(OmegaConf.to_container(cfg.data, resolve=True))
validation_dataset_sets, validation_dataset_cfgs = parse_validation_dataset_cfgs(data_cfg)

# Build dataloaders
train_dataloader, validation_dataloaders = setup_dataloaders_single_rank(
    data_root=cfg.paths.data_dir,
    dataset_collection=dataset_collection,
    dataloader_cfg=cfg.data.dataloader,
    train_dataset_cfg=cfg.data.train_dataset,
    pretraining_datasets=[],
    validation_dataset_cfgs=validation_dataset_cfgs,
    validation_dataset_sets=validation_dataset_sets,
)

# Save initial RNG states for reproducibility
initial_dl_rng_states = None
if fix_eval_rng_state:
    initial_dl_rng_states = {}
    initial_dl_rng_states["train"] = train_dataloader.get_state()
    for val_dl_key, val_dl in validation_dataloaders.items():
        val_dl_state = None
        if val_dl is not None:
            val_dl_state = val_dl.get_state()
        initial_dl_rng_states[val_dl_key] = val_dl_state

#### Let's look at a batch...

In [ ]:
# Get example batch
example_batch = next(iter(train_dataloader))
# Reset dataloader to start. NOTE: Dataloaders maintain state accross calls to `__iter__`
train_dataloader.reset_state()

session_key, batch_data = example_batch
responses = batch_data['responses'] # [batch, time, neurons]
response_timestamps = batch_data['timestamps']['responses'] # [batch, time]
screen = batch_data['screen'] # [batch, channels, time, height, width]
screen_timestamps = batch_data['timestamps']['screen'] # [batch, time]
eye_tracker = batch_data['eye_tracker'] # [batch, time, 4] (i.e. pupil x, y, dx, dy)
eye_tracker_timestamps = batch_data['timestamps']['eye_tracker'] # [batch, time]
treadmill = batch_data['treadmill'] # [batch, time, 1] (i.e. treadmill speed)
treadmill_timestamps = batch_data['timestamps']['treadmill'] # [batch, time]

print(f"Session key: {session_key}")
print(f"Responses shape: {responses.shape}")
print(f"Response timestamps shape: {response_timestamps.shape}")
print(f"Screen shape: {screen.shape}")
print(f"Screen timestamps shape: {screen_timestamps.shape}")
print(f"Eye tracker shape: {eye_tracker.shape}")
print(f"Eye tracker timestamps shape: {eye_tracker_timestamps.shape}")
print(f"Treadmill shape: {treadmill.shape}")
print(f"Treadmill timestamps shape: {treadmill_timestamps.shape}")

assert torch.equal(eye_tracker_timestamps, treadmill_timestamps)

In [ ]:
fig, axs = plt.subplots(1,1, figsize=(8,5))

n_samples = 60
n_neurons = 50
for n in range(n_neurons):
    t = np.arange(n, n_samples + n) # x-axis offset for each neuron
    responses = batch_data["responses"][0,:,n] + 0.1*n
    axs.plot(t, responses, color="black", alpha=0.5)

# Define points
point_a = (120, 1)
point_b = (130, 2)
axs.annotate('', xy=point_b, xytext=point_a,
            arrowprops=dict(arrowstyle='->', lw=2, color='k'))

# Add text above the arrow
mid_x = (point_a[0] + point_b[0]) / 2
mid_y = (point_a[1] + point_b[1]) / 2
axs.text(mid_x+5, mid_y + 0.5, 'Neurons', ha='center', va='bottom')

axs.set(xlabel="time in samples @ 30 Hz",
       ylabel="Neuronal activity",
       title="Neuronal activity for a sub-population of neurons")

### Model

#### Session Metadata

Collect metadata for each session in the dataset to use for initializing the model.

In [ ]:
session_map = SessionMap()
for dataset in train_dataloader.dataset.datasets:
    session_key = dataset.data_key
    n_neurons = dataset._experiment.devices["responses"].n_signals
    # NOTE: Mouse sess has format "XXXXX-M-D", where XXXXX animal ID, M-D is date
    animal_id = session_key.split('-')[0]
    # All sessions will be on the one notebook rank
    on_rank = True
    # Add metadata to session map
    session_meta = SessionMetadata(
        n_neurons=n_neurons,
        animal_id=animal_id,
        on_rank=on_rank,
    )
    session_map.update(session_key, session_meta)
pprint(session_map)

#### Instantiate model config dataclass

In [ ]:
model_config: ModelArgs = hydra.utils.instantiate(cfg.model)
pprint(model_config)

#### Build and initialize model

**Note:** We initialize model weights directly on the device (see discussion in [lightning docs](https://lightning.ai/docs/pytorch/stable/advanced/model_parallel/fsdp.html#speed-up-model-initialization))

In [ ]:
with torch.device(device):
    model: Model = build_model(model_config, session_map)
    # Should be a no-op with context manager, but just in case...
    model = model.to(device)
print_model_summary(model, max_depth=2)

#### Compile Model

**Note:** See discussion on order of compile/device placement/ddp [here](https://discuss.pytorch.org/t/torch-compile-before-or-after-cuda/176031)

In [ ]:
# TODO: You can enable/disable compile with this flag
COMPILE_ENABLED = True

# More fine-grained control over compile options can be done by further modifying the config
compile_cfg = deepcopy(cfg.trainer.compile)
compile_cfg.enabled = COMPILE_ENABLED
pprint(omegaconf_to_dict(compile_cfg))

In [ ]:
# Compile model
model = compile_model(model, cfg.trainer.compile)
# NOTE: Because we don't compile in place, the new `model` reference is actually an
#  `OptimizedModule`, which has `_orig_mod` prepended to all sub-module names.
pprint(model)

### Checkpointer

In [ ]:
# Get checkpointer config
ckpt_cfg = cfg.trainer.checkpointer
pprint(omegaconf_to_dict(ckpt_cfg))
# Instantiate checkpointer
checkpointer: ModelCheckpoint = hydra.utils.instantiate(ckpt_cfg)

# Load checkpoint
start_epoch, global_step = checkpointer.load_checkpoint(
    rank=0,
    world_size=1,
    model=model,
    train_dataloader=train_dataloader,
    val_dataloaders=validation_dataloaders,
    checkpoint_dir=cfg.ckpt_path,
    strict=False,
    load_random_states=False,
    load_weights_only=True,
)

### Choose a dataloader

In [ ]:
val_keys = list(validation_dataloaders.keys())
print(f"Avaliable dataloader keys:\n\ttrain\n" + "\n".join(f"\t{k}" for k in val_keys))
# TODO: Choose which dataloader here
VAL_KEY = "validation"

if VAL_KEY == "train":
    val_dataloader = train_dataloader
else:
    val_dataloader = validation_dataloaders[VAL_KEY]

### Select masks

##### What is a "Masking Strategy"?

We consider the activity/stimuli/behavior tensors for a context-window "block", and use
the masking strategy to define the boundaries on visible / hidden regions. Visible 
regions will be passed to the model. "reconstructed" regions serve as targets.

- **Responses**

```
                                num_samples_per_block
                                <---------------------------------------------------------------> 
                                        ┌> response_context_start_idx
                                        : max_response_context_samples
                                        :<---------------------------------------->:
                                        :   ┌> prefix_start_idx                    :
                                        :   : prefix_len                           :
                                        :   :<----------------------------->:      : 
                                        :   : full_population_prefix_len    :      :   
                                        :   :<---------->                   :      :
               A            A  ┌────────────┬──────────────────────-─–─–––––┬─────────–───────────┐
               │  n_visible │  │        :   │ xxxxxxxxxx : xxxxxxxxxxxxxxxx │      :              │
               │  _neurons  │  │        :   │ xxxxxxxxxx : x    prefix    x │      :              │
               │            V  │        :   │ x full_  x : xxxxxxxxxxxxxxxx │      :              │
 neural_       │               │        :   │ x popula x ┌──────────┬───-––─┴—————————————————————┼
population_    │            A  │        :   │ x  tion_ x │          │ xxxxxxxxxxxxxxxxxxxxxxxxxxx │
  size         │            │  │        :   │ x prefix x │          │ xxxxxxxxxxxxxxxxxxxxxxxxxxx │ 
               │  n_max_rec │  │        :   │ xxxxxxxxxx │          │ x          suffix         x │
               │  onstructe │  │        :   │ xxxxxxxxxx │          │ xxxxxxxxxxxxxxxxxxxxxxxxxxx │
               │  d_neurons V  │        :   | xxxxxxxxxx │          | xxxxxxxxxxxxxxxxxxxxxxxxxxx │
               V               └────────────┴────────────┴──────────┴───––––––-─––────────────────┘
                                                                    : <--------------------------->
                                                                    :          suffix_len
                                                                    └> suffix_start_idx
```

- **Video:**                              
   
```
                                                        ┌> n_visible_frames_start_idx
                                                        : n_visible_frames
                                                        :<----------------------->:
                               ┌────────────────────────┬──────────-─–─––––───────┬──–────────────┐
                               │                        │xxxxxxxxxxxxxxxxxxxxxxxxx│               │
                               └────────────────────────┴─────────────────––––––-─┴–───────–──────┘
```
   
- **Behavior**

```
                               behavior_encoded (True/False)     behavior_reconstructed (True/False)
                                      ┌──┐                                ┌──┐
                                      └──┘                                └──┘
```

**Key Parameters:**
- n_visible_neurons: Number of neurons visible to encoder (population masking)
- prefix_start_idx: Start position of visible prefix samples  
- prefix_len: Length of visible prefix samples (causal masking)
- full_population_prefix_len: Portion of prefix for which all neurons are visible,
    irrespective of `n_visible_neurons`.
- response_context_start_idx: Start position of "response context window". Response
    context window is used to determine which samples are available for encoding, and
    therefore which "latents" must be made available. The prefix region must be a subset
    of the response context window. See :mod:`omnimouse.modeling.model` for more details.
- max_response_context_samples: Maximum samples in response context
- max_n_reconstructed_neurons: Upper bound on number of neurons for reconstruction.
- suffix_start_idx: Start position of reconstruction period
- suffix_len: Length of reconstruction period
- n_visible_frames_start_idx: Start position of visible video frames
- n_visible_frames: Number of visible video frames
- behavior_encoded/behavior_reconstructed: Behavior masking controls
- weight: Sampling probability weight

**Common Configurations**:
- Population masking: n_visible_neurons < total, prefix_len = total
- Causal masking: n_visible_neurons = total, prefix_len < total  
- Two-stage causal: full_population_prefix_len > 0
- With reconstruction: max_n_reconstructed_neurons > 0

    Video masking uses causal approach starting from n_visible_frames_start_idx.
    Behavior masking is binary: either encoded or reconstructed, not both.

**Examples:**
```python
    # Basic population masking
    MaskingStrategy(n_visible_neurons=512)
    
    # Causal masking with custom start
    MaskingStrategy(prefix_start_idx=10, prefix_len=20)
    
    # Two-stage causal with reconstruction
    MaskingStrategy(
        n_visible_neurons=256,
        prefix_len=16, 
        full_population_prefix_len=8,
        max_n_reconstructed_neurons=128
    )
```

##### Use masks from the config

In [ ]:
# Get eval masking strategies for validation
val_masking_strategies = cfg.evaluation.get(f"{VAL_KEY}_masking_strategies", cfg.evaluation.default_masking_strategies)

##### Or you can build your own!

In [ ]:
# TODO: Add your own masking strategies here
val_masking_strategies = {
    "causal_response_behavior": MaskingStrategy(
        n_visible_neurons=4096,
        prefix_start_idx=0,
        prefix_len=20,
        full_population_prefix_len=0,
        response_context_start_idx=0,
        max_response_context_samples=60,
        max_n_reconstructed_neurons=4096,
        suffix_start_idx=25,
        suffix_len=30,
        n_visible_frames_start_idx=0,
        n_visible_frames=0,
        behavior_encoded=True,
        behavior_reconstructed=False,
        weight=1.0,
    )
}

##### Visualize the eval masking strategies

In [ ]:
SHOW_HTML_INLINE = False

masking_html = visualize_masking_strategies(
    val_masking_strategies,
    cfg.model.num_samples_per_block,
    cfg.model.num_frames_per_block,
    cfg.model.get("max_population_size"),
    cfg.model.get("n_neurons_multiple_of"),
    cfg.model.get("interpolation_buffer"),
    cfg.model.get("skip_n_samples"),
    cfg.model.get("response_stride_samples"),
    cfg.model.get("response_window_size_samples"),
    cfg.get("max_num_strategies"),
    output_file=output_dir / "validation_strategies_interactive.html",
    title="Validation Masking Strategies",
)

if SHOW_HTML_INLINE:
    display(HTML(masking_html))

### Run eval

In [ ]:
# Whether to predict full unmasked population, or just random subset or 4096 neurons
ALL_NEURONS_OVERRIDE = False
# TODO: Enable/disable autocasting
AUTOCAST_ENABLED = True
# TODO: Set these to only val on a few batches for debugging
LIMIT_VAL_BATCHES = None

# Enable autocasting
dtype = torch.bfloat16 if AUTOCAST_ENABLED else torch.float32
# Set model to eval mode
model.eval()

print(f"Validating at step {global_step} (epoch {start_epoch})")
# Run validation on "validation" set
val_loss, val_metrics = run_evaluation(
    model, val_dataloader, AUTOCAST_ENABLED, device, LIMIT_VAL_BATCHES,
    val_masking_strategies, 0, VAL_KEY, initial_rng_state,
    initial_dl_rng_states, all_neurons_override=ALL_NEURONS_OVERRIDE,
) if val_dataloader is not None else (None, None)
# Add cross rank / session averages
val_loss = add_overall_metrics(val_loss, session_map)
val_metrics = add_overall_metrics(val_metrics, session_map)
# Log to loggers
if logger is not None:
    log_to_loggers(
        logger,
        val_loss,
        prefix="validation",
        step=global_step,
    )
    log_to_loggers(logger, val_metrics, step=global_step, prefix="validation")
    log_to_loggers(logger, val_loss, step=global_step, prefix="validation")
    print(f"Check out your results here! ({os.path.join(
        logger._tracking_uri,'#', 'experiments', logger.experiment_id, 'runs', logger.run_id
    )}")
